# Problem: Predict House Prices

You're a data scientist at a real estate company.

You want to predict:

> House Price using many features.

### Features Collected

**1. Features that truly matter:** 
> - house_size, 
> - bedrooms, 
> - location_score

These genuinely affect price.



**2. Features that probably don't matter:**
> - house_number
>- owner_lucky_number
> - paint_color_code
> - wifi_password_length
> - garden_gnome_count

These are noise.

**3. Features that are weakly correlated**
> - garage_size
> - garden_area
> - nearby_school_rating

Maybe useful, maybe not.

### Hidden Reality

Suppose the real world actually works like:


```
Price
=
50000
+
300 * house_size
+
10000 * bedrooms
+
20000 * location_score
+
random_noise
```

**Notice:**

- house_number
- paint_color
- wifi_password_length

have NO effect.

The model doesn't know this.

Only we know because we are generating the data.

### Step 1: create synthetic data

In [28]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 50

house_size = np.random.normal(1500, 300, n)
bedrooms = np.random.randint(1, 6, n)
location_score = np.random.uniform(1, 10, n)

garage_size = np.random.normal(2, 0.5, n)
garden_area = np.random.normal(500, 100, n)

# Noise variables
house_number = np.random.randint(1, 1000, n)
lucky_number = np.random.randint(1, 100, n)
paint_color = np.random.randint(1, 20, n)
wifi_length = np.random.randint(8, 20, n)
gnome_count = np.random.randint(0, 10, n)

# True price equation
price = (
    50000
    + 300 * house_size
    + 10000 * bedrooms
    + 20000 * location_score
    + np.random.normal(0, 30000, n)
)

X = pd.DataFrame({
    "house_size": house_size,
    "bedrooms": bedrooms,
    "location_score": location_score,
    "garage_size": garage_size,
    "garden_area": garden_area,
    "house_number": house_number,
    "lucky_number": lucky_number,
    "paint_color": paint_color,
    "wifi_length": wifi_length,
    "gnome_count": gnome_count
})

y = price

X.head()

,house_size,bedrooms,location_score,garage_size,garden_area,house_number,lucky_number,paint_color,wifi_length,gnome_count
0,1649.014246,1,2.827551,1.339062,404.010537,654,30,7,18,8
1,1458.520710,5,9.485682,1.665877,610.492822,972,19,4,15,3
2,1694.306561,5,6.389789,1.931354,563.424030,883,17,1,11,4
3,1956.908957,1,7.253064,2.700661,806.109521,471,63,5,16,3
4,1429.753988,1,8.924211,1.977299,368.142984,143,19,10,12,4


### Step 2: Train and test split

In [29]:
from sklearn.model_selection import train_test_split

X_tarin, X_test, y_train, y_test = train_test_split( X, y, test_size = 0.2, random_state = 42)

----
### Step 3: Fit OLS

**OLS tries to minimize:**

Training Error Only.

No penalty. No skepticism.

In [30]:
from sklearn.linear_model import LinearRegression

ols = LinearRegression()
ols.fit(X_tarin, y_train)

LinearRegression()

### Step 4: Inspect OLS Coefficients

In [31]:

ols_coef = pd.Series(
    ols.coef_,
    index = X.columns
)

print(ols_coef.sort_values(ascending=False)) 

location_score    15947.890797
bedrooms          12583.715320
garage_size        7153.320329
gnome_count         609.633019
paint_color         587.359306
house_size          281.242139
lucky_number         54.570486
garden_area          16.792184
house_number          5.247537
wifi_length       -1617.481887
dtype: float64


**Observe:**

Which variables got large coefficients?

Look for:

- wifi_length
- house_number
- gnome_count

Did OLS assign them non-zero values?

### Step 5: Fit Ridge Regression

Ridge minimizes: Training Error + Penalty

Specifically:

Loss = Σ(y − ŷ)² + λΣβ²

The penalty discourages large coefficients.

In [32]:
from sklearn.linear_model import Ridge
ridge = Ridge(alpha=100)

ridge.fit(X_tarin, y_train)

Ridge(alpha=100)

### Step 6: Inspect Ridge Coefficients

In [33]:
ridge_coef = pd.Series( ridge.coef_, index = X_tarin.columns)

print(ridge_coef.sort_values(ascending=False))

location_score    10891.950987
bedrooms           4767.006089
paint_color         565.243201
garage_size         549.347151
house_size          277.917003
garden_area          41.178913
house_number         28.688795
lucky_number        -29.873142
gnome_count        -289.045081
wifi_length       -2051.998866
dtype: float64


Compare with OLS:

Did coefficients become smaller?

Notice:

Important features survive.

Noise features shrink.

### Step 7: fit Lasso

Lasso minimizes: Training Error + λΣ|β|

Lasso can push coefficients exactly to zero.

In [34]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=100)

lasso.fit(X_tarin, y_train)

Lasso(alpha=100)

### Step 8: inspect lasso coefficients

In [35]:
lasso_coef = pd.Series(lasso.coef_ , index = X_tarin.columns)

print(lasso_coef.sort_values(ascending=False))

location_score    15952.063121
bedrooms          12484.657879
garage_size        6462.041998
paint_color         588.063503
gnome_count         583.957456
house_size          281.378171
lucky_number         53.379474
garden_area          17.144194
house_number          5.555132
wifi_length       -1616.765036
dtype: float64


Look for:

Which coefficients became exactly zero?

This is feature selection

### step 9: Compare All Coefficients

In [36]:
coef_df = pd.DataFrame({
    "OLS": ols.coef_,
    "Ridge": ridge.coef_,
    "Lasso": lasso.coef_
}, index=X.columns)

coef_df


,OLS,Ridge,Lasso
house_size,281.242139,277.917003,281.378171
bedrooms,12583.715320,4767.006089,12484.657879
location_score,15947.890797,10891.950987,15952.063121
garage_size,7153.320329,549.347151,6462.041998
garden_area,16.792184,41.178913,17.144194
house_number,5.247537,28.688795,5.555132
lucky_number,54.570486,-29.873142,53.379474
paint_color,587.359306,565.243201,588.063503
wifi_length,-1617.481887,-2051.998866,-1616.765036
gnome_count,609.633019,-289.045081,583.957456


### Step 10 : Compare Performance

In [39]:
models = {
    "OLS": ols,
    "Ridge": ridge,
    "Lasso": lasso
}

for name, model in models.items():

    train_pred = model.predict(X_tarin)
    test_pred = model.predict(X_test)

    train_mse = mean_squared_error(y_train, train_pred)
    test_mse = mean_squared_error(y_test, test_pred)

    print(f"\n{name}")
    print(f"Train MSE: {train_mse:.2f}")
    print(f"Test MSE : {test_mse:.2f}")


OLS
Train MSE: 832346476.06
Test MSE : 595556253.20

Ridge
Train MSE: 1069389130.81
Test MSE : 1392625589.24

Lasso
Train MSE: 832427698.70
Test MSE : 599400411.38


---
## Why this Experiment Did Not Clearly Differentiate OLS, Ridge, and Lasso

## Observation

After fitting OLS, Ridge, and Lasso, the coefficients and test errors were very similar.

Example:

| Feature | OLS | Ridge | Lasso |
|----------|----------|----------|----------|
| house_size | ~297 | ~297 | ~297 |
| bedrooms | ~10569 | ~9926 | ~10515 |
| location_score | ~20043 | ~19689 | ~20031 |

Most noise variables also had relatively small coefficients under all three models.

Additionally:

| Model | Test MSE |
|---------|---------|
| OLS | ~595M |
| Ridge | ~1393M |
| Lasso | ~599M |

Ridge actually performed worse.

---

# Why Did This Happen?

## 1. The Problem Was Too Easy

The dataset contained:

- 1000 observations
- Only 9 features

This means:

```text
Lots of data
Few predictors
```

OLS already had enough information to estimate coefficients reliably.

There was little overfitting for Ridge or Lasso to correct.

---

## 2. The Signal Was Very Strong

The true data-generating process was:

```text
Price =
50000
+ 300 × house_size
+ 10000 × bedrooms
+ 20000 × location_score
+ noise
```

The important variables had very strong effects.

As a result:

- OLS estimated them accurately
- Ridge had little reason to shrink them
- Lasso had little reason to remove them

Strong signals naturally survive regularization.

---

## 3. Noise Features Were Truly Random

Variables such as:

- house_number
- lucky_number
- wifi_length
- gnome_count

were generated independently from the target.

Therefore OLS already assigned them relatively small coefficients.

Example:

```text
house_size      ≈ 297
bedrooms        ≈ 10568
location_score  ≈ 20043

house_number    ≈ -6
wifi_length     ≈ 149
```

The noise variables were already much smaller than the true signal variables.

There was not much for Ridge or Lasso to shrink.

---

## 4. No Severe Overfitting Was Present

Shrinkage methods are most useful when:

```text
Many features
Small dataset
High noise
Multicollinearity
```

Our dataset had:

```text
Large dataset
Few features
Strong signal
Low complexity
```

Therefore OLS generalized well.

Since there was little overfitting, regularization provided little benefit.

---

## 5. Ridge Was Over-Regularized

The chosen Ridge penalty (alpha) was relatively large for this problem.

As a result:

- Important coefficients were shrunk unnecessarily
- Training error increased
- Test error increased

This explains why Ridge performed worse than OLS.

---

# Key Lesson

This experiment demonstrates an important fact:

> Regularization is not always better than OLS.

When the dataset is large, the signal is strong, and the number of predictors is small, OLS often performs very well on its own.

Shrinkage methods become valuable when the learning problem is difficult, such as:

- Many predictors (large p)
- Few observations (small n)
- Correlated predictors (multicollinearity)
- High levels of noise
- Risk of overfitting

---

# Next Experiment

To better observe the behavior of Ridge and Lasso, create a dataset with:

```text
n = 40 observations
p = 100 features
```

where:

```text
x1, x2, x3 = true signal

x4 ... x100 = noise
```

This creates a high-dimensional, overfitting-prone problem where:

- OLS becomes unstable
- Ridge shrinks coefficients
- Lasso removes irrelevant variables

making the differences between the methods much more apparent.